Imports

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
dados = pd.read_json("/content/drive/MyDrive/Colab Notebooks/Alura Challenge DS 1/Telco-Customer-Churn-limpeza.json")
dados.head()

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,Charges.Daily,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,No,Yes,Yes,No,One year,Yes,65.922222,Mailed check,65.6,593.30
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,Yes,Month-to-month,No,60.266667,Mailed check,59.9,542.40
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,Yes,No,No,No,Month-to-month,Yes,70.212500,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,No,Yes,Yes,Month-to-month,Yes,95.219231,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,Yes,Yes,No,Month-to-month,Yes,89.133333,Mailed check,83.9,267.40


In [ ]:
dados.drop(['customerID', 'Charges.Total'], axis=1, inplace=True)

In [ ]:
for i in dados.select_dtypes(include=['object']).columns:
    if len(dados[i].unique()) > 2:
       print(f"{i}: {dados[i].unique()}")

MultipleLines: ['No' 'Yes' 'No phone service']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['Yes' 'No' 'No internet service']
StreamingTV: ['Yes' 'No' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['One year' 'Month-to-month' 'Two year']
PaymentMethod: ['Mailed check' 'Electronic check' 'Credit card (automatic)'
 'Bank transfer (automatic)']


In [ ]:
colunas = ['PaymentMethod', 'Contract', 'InternetService']
dados2 = dados.drop(colunas, axis=1)
dados2.columns

Index(['Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'Charges.Daily', 'Charges.Monthly'],
      dtype='object')

In [ ]:
dicionario = {'No internet service':0,
              'No phone service': 0,
              'No': 0,
              'Yes': 1,
              'Male':0,
              'Female':1}

In [ ]:
dados2 = dados2.replace(dicionario)
dados2.head()

<ipython-input-7-3309cdce4b73>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dados2 = dados2.replace(dicionario)


,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,Charges.Daily,Charges.Monthly
0,0,1,0,1,1,9,1,0,0,1,0,1,1,0,1,65.922222,65.6
1,0,0,0,0,0,9,1,1,0,0,0,0,0,1,0,60.266667,59.9
2,1,0,0,0,0,4,1,0,0,0,1,0,0,0,1,70.212500,73.9
3,1,0,1,1,0,13,1,0,0,1,1,0,1,1,1,95.219231,98.0
4,1,1,1,1,0,3,1,0,0,0,0,1,1,0,1,89.133333,83.9


In [ ]:
ohe = OneHotEncoder(dtype=int, sparse_output=False)

# Aplicar OneHotEncoder nas colunas categóricas
colunas_ohe = ohe.fit_transform(dados[colunas])

# Criar um novo DataFrame com as colunas geradas
df_ohe = pd.DataFrame(colunas_ohe, columns=ohe.get_feature_names_out(colunas))

# Concatenar com o DataFrame original
dados3 = pd.concat([dados2, df_ohe], axis=1)
dados3

,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,...,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No
0,0,1,0,1,1,9,1,0,0,1,...,0,0,0,1,0,1,0,1,0,0
1,0,0,0,0,0,9,1,1,0,0,...,0,0,0,1,1,0,0,1,0,0
2,1,0,0,0,0,4,1,0,0,0,...,0,0,1,0,1,0,0,0,1,0
3,1,0,1,1,0,13,1,0,0,1,...,0,0,1,0,1,0,0,0,1,0
4,1,1,1,1,0,3,1,0,0,0,...,0,0,0,1,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,1,0,0,0,13,1,0,1,0,...,0,0,0,1,0,1,0,1,0,0
7039,1,0,0,1,0,22,1,1,0,0,...,0,0,1,0,1,0,0,0,1,0
7040,0,0,0,0,0,2,1,0,0,1,...,0,0,0,1,1,0,0,1,0,0
7041,0,0,0,1,1,67,1,0,1,0,...,0,0,0,1,0,0,1,1,0,0


Smote

In [ ]:
X = dados3.drop(['Churn'], axis=1).dropna()
y = dados3.loc[X.index, 'Churn']

In [ ]:
SEED = 42
sm = SMOTE(random_state=SEED)
X_res, y_res = sm.fit_resample(X, y)
dados4 = pd.concat([pd.DataFrame(X_res, columns=X.columns), pd.DataFrame(y_res, columns=['Churn'])], axis=1)
dados4

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,Churn
0,1,0,1,1,9,1,0,0,1,0,...,0,0,1,0,1,0,1,0,0,0
1,0,0,0,0,9,1,1,0,0,0,...,0,0,1,1,0,0,1,0,0,0
2,0,0,0,0,4,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,1
3,0,1,1,0,13,1,0,0,1,1,...,0,1,0,1,0,0,0,1,0,1
4,1,1,1,0,3,1,0,0,0,0,...,0,0,1,1,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10321,0,0,0,0,4,1,1,0,1,0,...,0,1,0,1,0,0,0,1,0,1
10322,1,0,0,0,3,1,0,0,0,0,...,0,1,0,1,0,0,0,1,0,1
10323,0,0,0,0,49,1,1,0,0,0,...,0,1,0,1,0,0,0,1,0,1
10324,0,0,0,0,18,1,0,0,0,0,...,0,1,0,1,0,0,0,1,0,1


In [ ]:
#salvando
dados4.to_json("/content/drive/MyDrive/Colab Notebooks/Alura Challenge DS 1/Telco-Customer-Churn-balanceamento.json")

#Criando Modelos

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Criando os modelos
svc_model = SVC(random_state=42)
dt_model = DecisionTreeClassifier(random_state=42)
rf_model = RandomForestClassifier(random_state=42)

# Treinando os modelos
svc_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [ ]:
# Fazendo previsões
y_pred_svc = svc_model.predict(X_test)
y_pred_dt = dt_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Função para calcular métricas
def avaliar_modelo(nome, y_test, y_pred):
    print(f"📊 {nome}:")
    print(f"Acurácia: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precisão: {precision_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred, pos_label=1):.4f}")
    print(f"F1-score: {f1_score(y_test, y_pred, pos_label=1):.4f}")
    print("-" * 40)

# Avaliando cada modelo
avaliar_modelo("SVC", y_test, y_pred_svc)
avaliar_modelo("Decision Tree", y_test, y_pred_dt)
avaliar_modelo("Random Forest", y_test, y_pred_rf)

📊 SVC:
Acurácia: 0.7953
Precisão: 0.6991
Recall: 0.4037
F1-score: 0.5119
----------------------------------------
📊 Decision Tree:
Acurácia: 0.7235
Precisão: 0.4802
Recall: 0.4866
F1-score: 0.4834
----------------------------------------
📊 Random Forest:
Acurácia: 0.7925
Precisão: 0.6434
Recall: 0.4920
F1-score: 0.5576
----------------------------------------


#Melhorando o Modelo

In [ ]:
X = dados.drop(['Churn'], axis=1)
y = dados['Churn']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=SEED)

In [ ]:
n_estimators = np.arange(100, 200, step=20)
criterion = ["gini", "entropy"]
max_features = ["auto", "log2"]
max_depth = list(np.arange(2, 10, step=2))
min_samples_split = np.arange(2, 10, step=2)
min_samples_leaf = [2, 4]
bootstrap = [True, False]

parameters = {
    "n_estimators": n_estimators,
    "criterion": criterion,
    "max_features": max_features,
    "max_depth": max_depth,
    "min_samples_split": min_samples_split,
    "min_samples_leaf": min_samples_leaf,
    "bootstrap": bootstrap,
}